In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from mnist import get_mnist_dataset

import pandas as pd

import holoviews as hv
hv.extension("bokeh")

In [ ]:
X_train, Y_train, X_test, Y_test = get_mnist_dataset()

In [ ]:
def train_model(epochs, batch_size, validation_split):
    from mini_dl.nn.model import Sequential
    from mini_dl.nn.layer import Dense, ReLU, SoftMax
    from mini_dl.optim.optimizer import SGDOptimizer, Adam
    from mini_dl.nn.loss import SumOfSquares, CrossEntropy
    
    model = Sequential([
        Dense(784, 250),
        ReLU(),
        Dense(250, 250),
        ReLU(),
        Dense(250, 10),
    ])

    model.compile_(optimizer=Adam(), loss=CrossEntropy())

    hist = model.fit(X_train, Y_train, epochs=40, batch_size=128, validation_split=0.3)
    return hist

In [ ]:
param_sets = pd.DataFrame({
    "epochs": 40,
    "batch_size": ([128] * 5) + ([64] * 5) + ([32] * 5),
    "valid_split": [0.1, 0.2, 0.3, 0.4, 0.5] * 3
})
param_sets

In [ ]:
histories = [train_model(*params) for _, params in param_sets.iterrows()]

In [ ]:
train_losses = [h["train_loss"] for h in histories]
valid_losses = [h["valid_loss"] for h in histories]
param_sets["train_loss"] = train_losses
param_sets["valid_loss"] = valid_losses

In [ ]:
param_sets.head()

In [ ]:
def to_Curve(row):
    _, batch_size, valid_split, train_loss, valid_loss = row

    colors = {128: "red", 64: "green", 32: "blue"}

    return (
        hv.Curve(train_loss, label=f"{batch_size=} {valid_split=}").opts(
            color=colors[batch_size],
            line_dash="dashed",
            alpha=valid_split,
        )
        * hv.Curve(valid_loss, label=f"{batch_size=} {valid_split=}").opts(
            color=colors[batch_size],
            alpha=valid_split,
            show_legend=True,
        )
    ).opts(legend_opts={"click_policy": "hide"})

In [ ]:
# Query by split size, batch size, etc...
curves = param_sets.loc[
    param_sets["valid_split"] == 0.3
].apply(to_Curve, axis=1).to_list()

hv.Overlay(curves).opts(width=600, height=600)

In [ ]:
# param_sets.to_feather(Path.cwd().parent.parent / "artifacts" / "validation_set.arrow")